In [ ]:
import os
import subprocess
import torch
import gc
import shutil
import time
import re
import glob

# Chạy đoạn này để GPU luôn hoạt động nhẹ, tránh bị Colab báo idle
import threading
def keep_gpu_busy():
    if not torch.cuda.is_available(): # Kiểm tra xem GPU có sẵn sàng không
        print("⚠️ Không tìm thấy GPU. Bỏ qua tác vụ giữ GPU hoạt động.")
        return
    print("🔋 Bắt đầu giữ GPU hoạt động ngầm...")

    a = torch.ones(1000, 1000).cuda() # Tạo một tensor nhỏ trên GPU
    while True:
        a = a @ a  # Thực hiện phép nhân ma trận đơn giản liên tục
        time.sleep(120) # Nghỉ 120s (2 phút) rồi làm tiếp để không tốn quá nhiều tài nguyên
t = threading.Thread(target=keep_gpu_busy) # Chạy trong luồng riêng biệt (background thread)
t.daemon = True
t.start()

In [ ]:
from google.colab import drive

# --- CẤU HÌNH ---
FOLDER_NAME = "Survival"
DRIVE_ROOT_PATH = "/content/drive/MyDrive/CharenjiZukan/2603"

# Đường dẫn nguồn (Drive) và đích (Local)
SOURCE_PARENT_PATH = os.path.join(DRIVE_ROOT_PATH)
LOCAL_PARENT_PATH = os.path.join("/content", FOLDER_NAME)
# 1. MOUNT DRIVE
drive.mount('/content/drive', force_remount=True)
# 2. LÀM SẠCH THƯ MỤC LOCAL CŨ
# Dùng subprocess rm -rf để xóa sạch thư mục cũ nếu có
subprocess.run(["rm", "-rf", LOCAL_PARENT_PATH], check=False)
subprocess.run(["rm", "-rf", "/content/assets"], check=False)
# Tạo lại thư mục cha rỗng ở local
os.makedirs(LOCAL_PARENT_PATH, exist_ok=True)

# Copy thư mục assets từ Drive về Local
print("📥 [COPY] Đang tải thư mục assets...")
subprocess.run(["cp", "-r", "/content/drive/MyDrive/CharenjiZukan/0 - assets", "/content/assets"], check=False)
print("✅ Hoàn tất copy thư mục assets!")

# 3. DUYỆT VÀ COPY THEO ĐIỀU KIỆN
if os.path.exists(SOURCE_PARENT_PATH):
    print(f"🚀 Bắt đầu đồng bộ từ: {SOURCE_PARENT_PATH}")
    # Lấy danh sách các thư mục con (Chapter)
    subfolders = sorted([f for f in os.listdir(SOURCE_PARENT_PATH)
                         if os.path.isdir(os.path.join(SOURCE_PARENT_PATH, f))])

    for item_name in subfolders:
        # Định nghĩa các đường dẫn cần thiết cho Item hiện tại
        source_item_path = os.path.join(SOURCE_PARENT_PATH, item_name)
        local_item_path = os.path.join(LOCAL_PARENT_PATH, item_name)
        # File/Folder cần kiểm tra trên Drive
        drive_assets_folder = os.path.join(source_item_path, "assets")
        # --- KIỂM TRA ĐIỀU KIỆN ---
        # Kiểm tra xem trên Drive đã có thư mục assets chưa
        if os.path.exists(drive_assets_folder) and os.path.isdir(drive_assets_folder):
            print(f"⏩ [BỎ QUA] Đã có folder 'assets': {item_name}")
            continue # Nhảy sang vòng lặp kế tiếp
        else:
            print(f"📥 [COPY] Đang tải dữ liệu nguồn: {item_name}")
            # 1. Tạo thư mục đích tại Local (tương đương lệnh: mkdir -p "path")
            subprocess.run(["mkdir", "-p", local_item_path], check=True)
            # 2. Tìm và Copy file mp4
            # Sử dụng glob để quét tất cả các file có đuôi .mp4 trong thư mục
            mp4_files = glob.glob(os.path.join(source_item_path, "*.mp4"))
            if mp4_files:
                # Lấy file mp4 đầu tiên tìm thấy
                source_mp4_path = mp4_files[0]
                subprocess.run(["cp", source_mp4_path, local_item_path], check=True)
                print(f"  ✅ Đã copy: {os.path.basename(source_mp4_path)}")
            else:
                print(f"  ⚠️ Cảnh báo: Không tìm thấy file video (.mp4) tại {item_name}")
    print("\n✅ Hoàn tất quá trình copy!")
else:
    print(f"❌ Không tìm thấy thư mục gốc trên Drive: {SOURCE_PARENT_PATH}")

In [ ]:
!apt-get -y install fonts-noto-cjk
!fc-cache -fv

# Cài đặt uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] += ':/root/.local/bin'

# Clone project (Private Repository với Secrets)
from google.colab import userdata
token = userdata.get('github_token')
!git clone https://{token}@github.com/ThanhVoKim/CharenjiZukan.git /content/CharenjiZukan

!pip install -q pyyaml pytest

In [ ]:
%cd /content/CharenjiZukan
!uv pip install -e .

# Cài đặt rubberband-cli (cần cho time-stretch)
!apt-get install -y rubberband-cli

# Tạo virtual environment và cài whisperx
!uv venv
# !uv pip install -p .venv/bin/python whisperx
!uv pip install -p .venv/bin/python pyrubberband

# Cài CUDA dependencies
# !apt install libcudnn8 libcudnn8-dev -y

# Set environment variables
# %env MPLBACKEND=agg
# %env TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=true
# %env LD_LIBRARY_PATH=/usr/lib64-nvidia:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib/

# !uv run whisperx test.mp3 --language en --hf_token {hf_token} --diarize  --model large-v2 --align_model WAV2VEC2_ASR_LARGE_LV60K_960H

In [ ]:
# @title 🎯 BATCH SCRIPT: XỬ LÝ HÀNG LOẠT THƯ MỤC SURVIVAL (RENDER 0.65X & STT)

FOLDER_NAME = "/content/Survival"
DRIVE_ROOT_PATH = "/content/drive/MyDrive/CharenjiZukan/2603"
PYVIDEOTRANS_DIR = "/content/pyvideotrans"
CLI_CMD = ["uv", "run", "cli.py"]

print("🚀 BẮT ĐẦU XỬ LÝ HÀNG LOẠT...")

if os.path.exists(FOLDER_NAME):
    # Lấy danh sách các thư mục con
    subfolders = sorted([f for f in os.listdir(FOLDER_NAME) if os.path.isdir(os.path.join(FOLDER_NAME, f))])

    for item_name in subfolders:
        print(f"\n" + "="*50)
        print(f"📦 Đang xử lý thư mục: {item_name}")

        item_path = os.path.join(FOLDER_NAME, item_name)
        assets_path = os.path.join(item_path, "assets")
        drive_assets_path = os.path.join(DRIVE_ROOT_PATH, item_name, "assets")
        os.makedirs(assets_path, exist_ok=True)

        # Đường dẫn video đích và file srt đích
        slow_video_name = f"{item_name}.wav"
        slow_video_path = os.path.join(item_path, slow_video_name)
        dest_srt_path = os.path.join(assets_path, f"raw-{item_name}.srt")

        if os.path.exists(dest_srt_path):  # BỎ QUA NẾU ĐÃ CÓ PHỤ ĐỀ TRONG ASSETS
            print(f"⏩ Đã có phụ đề STT '{dest_srt_path}'. Bỏ qua!")
            continue

        if not os.path.exists(slow_video_path):
            mp4_files = glob.glob(os.path.join(item_path, "*.mp4")) # Tìm video gốc (Lấy file .mp4 đầu tiên không trùng tên với video 0.65x)
            original_video = None
            for mp4 in mp4_files:
                if os.path.basename(mp4) != slow_video_name:
                    original_video = mp4
                    break

            if original_video:
                print(f"⏳ [1/3] Đang render giảm tốc độ (0.65x) từ: {os.path.basename(original_video)}...")

                ffmpeg_cmd = [
                    "ffmpeg", "-y",
                    "-i", original_video,
                    "-vn",                                   # Bỏ hoàn toàn luồng video
                    "-filter:a", "atempo=0.65",              # Làm chậm âm thanh
                    "-ar", "16000",                          # Ép về 16kHz (chuẩn vàng cho các model STT)
                    "-ac", "1",                              # Chuyển về Mono (STT không cần Stereo)
                    "-c:a", "pcm_s16le",                     # Format WAV nguyên bản (không nén, cực nhanh)
                    slow_video_path
                ]
                subprocess.run(ffmpeg_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
                print(f"  ✅ Render thành công: {slow_video_name}")
            else:
                print(f"  ⚠️ Cảnh báo: Không tìm thấy video gốc để render!")
                continue # Chuyển sang thư mục tiếp theo
        else:
            print(f"⏩ [1/3] Video 0.65x đã tồn tại, bỏ qua render.")

        if os.path.exists(original_video):
            print(f"🎙️ [2/3] Đang chạy STT...")
            stt_cmd = CLI_CMD + [
                "--task", "stt",
                "--name", original_video,
                "--recogn_type", "1", "--model_name", "large-v2", "--cuda"
            ]
            # Chạy lệnh (Ép chạy tại thư mục pyvideotrans để tránh lỗi missing modules)
            subprocess.run(stt_cmd, cwd=PYVIDEOTRANS_DIR, check=False)

            print(f"🔍 [3/3] Đang tìm kiếm file kết quả...")
            # Theo chuẩn pyvideotrans, output folder thường bỏ dấu cách và ký tự đặc biệt
            formatted_basename = re.sub(r'[\s\. #*?!:"]', '-', slow_video_name)
            srt_candidates = glob.glob(os.path.join(PYVIDEOTRANS_DIR, "output", f"*{formatted_basename}*", "*.srt"))

            if srt_candidates:
                latest_srt = max(srt_candidates, key=os.path.getctime) # Lấy file srt vừa mới được tạo ra gần nhất
                subprocess.run(["mv", latest_srt, dest_srt_path], check=True) # Cut file (Move) bằng lệnh Linux
                # Đọc file, dọn dẹp thẻ [spk] và ghi đè lại
                with open(dest_srt_path, "r", encoding="utf-8") as f:
                    text = f.read()
                clean_text = re.sub(r'\[spk\d+\]', '', text)
                with open(dest_srt_path, "w", encoding="utf-8") as f:
                    f.write(clean_text)

                print(f"  ✅ Đã CUT file, dọn sạch rác [spk] và lưu tại: assets/raw-{item_name}.srt")
                # Lệnh Linux tạo folder đích (-p tự động tạo các folder cha nếu chưa có)
                subprocess.run(["mkdir", "-p", drive_assets_path], check=True)
                # Dùng lệnh Linux cp để copy chính xác 3 file (Dùng shell=True để nhận wildcard và nhiều param)
                cp_command = f"cp '{dest_srt_path}' '{drive_assets_path}/' 2>/dev/null || true"
                subprocess.run(cp_command, shell=True)
            else:
                print(f"  ❌ LỖI: Không tìm thấy file phụ đề đầu ra cho {item_name}.")

print("\n🎉 HOÀN TẤT TOÀN BỘ QUÁ TRÌNH!")

In [ ]:
# @title 🎯 BATCH SCRIPT: DỊCH PHỤ ĐỀ (STS), LỒNG TIẾNG (TTS) VÀ BACKUP LÊN DRIVE
# --- CẤU HÌNH ---
FOLDER_NAME = "/content/Survival"
DRIVE_ROOT_PATH = "/content/drive/MyDrive/CharenjiZukan/2603"
PYVIDEOTRANS_DIR = "/content/pyvideotrans"

if os.path.exists(FOLDER_NAME):
    # Lấy danh sách các thư mục con
    subfolders = sorted([f for f in os.listdir(FOLDER_NAME) if os.path.isdir(os.path.join(FOLDER_NAME, f))])

    for item_name in subfolders:
        print(f"\n" + "="*50)
        print(f"📦 Đang xử lý thư mục: {item_name}")

        assets_path = os.path.join(FOLDER_NAME, item_name, "assets")
        drive_assets_path = os.path.join(DRIVE_ROOT_PATH, item_name, "assets")

        # Định nghĩa sẵn các đường dẫn file
        raw_srt_path = os.path.join(assets_path, f"raw-{item_name}.srt")
        sts_srt_path = os.path.join(assets_path, f"sts-{item_name}.srt")
        tts_audio_path = os.path.join(assets_path, f"tts-{item_name}.m4a")

        # Kiểm tra điều kiện tiên quyết: Phải có file raw thì mới làm tiếp được
        if not os.path.exists(raw_srt_path):
            print(f"  ❌ Không tìm thấy file {raw_srt_path}. Bỏ qua thư mục này!")
            continue

        # ==========================================
        # BƯỚC 1: DỊCH PHỤ ĐỀ (STS)
        # ==========================================
        if not os.path.exists(sts_srt_path):
            print(f"🌐 [1/4] Đang dịch phụ đề sang tiếng Nhật...")
            sts_cmd = [
                "uv", "run", "cli.py",
                "--task", "sts",
                "--name", raw_srt_path,
                "--target_language_code", "ja",
                "--translate_type", "5"
            ]
            subprocess.run(sts_cmd, cwd=PYVIDEOTRANS_DIR, check=False)

            # Săn lùng file dịch vừa tạo (Linux mv command)
            formatted_basename = re.sub(r'[\s\. #*?!:"]', '-', f"raw-{item_name}.srt")
            sts_candidates = glob.glob(os.path.join(PYVIDEOTRANS_DIR, "output", f"*{formatted_basename}*", "*.srt"))

            if sts_candidates:
                latest_sts = max(sts_candidates, key=os.path.getctime)
                # Dùng Linux command để CUT (move) file
                subprocess.run(["mv", latest_sts, sts_srt_path], check=True)
                print(f"  ✅ Đã dịch và CUT file thành công: {sts_srt_path}")
            else:
                print(f"  ⚠️ Lỗi: Không tìm thấy file đầu ra STS.")
                continue # Dừng dây chuyền của thư mục này nếu lỗi
        else:
            print(f"⏩ [1/4] Đã có sẵn file sts.srt. Bỏ qua dịch.")

        # ==========================================
        # BƯỚC 2: LỒNG TIẾNG (TTS) CHO FILE STS
        # ==========================================
        if os.path.exists(sts_srt_path) and not os.path.exists(tts_audio_path):
            print(f"🎙️ [2/4] Đang lồng tiếng (TTS) với giọng ja-JP-KeitaNeural...")
            tts_cmd = [
                "uv", "run", "my_cli.py",
                "--name", sts_srt_path,
                "--tts_type", "0",
                "--voice_role", "ja-JP-KeitaNeural",
                "--target_language_code", "ja"
            ]
            subprocess.run(tts_cmd, cwd=PYVIDEOTRANS_DIR, check=False)

            # Săn lùng file audio vừa tạo (Bám theo tên file đầu vào mới)
            formatted_sts_basename = re.sub(r'[\s\. #*?!:"]', '-', f"sts-{item_name}.srt")
            tts_candidates = glob.glob(os.path.join(PYVIDEOTRANS_DIR, "output", f"*{formatted_sts_basename}*", "*.wav"))

            if tts_candidates:
                latest_tts = max(tts_candidates, key=os.path.getctime)
                ffmpeg_cmd = [ # Dùng FFmpeg để nén file và xuất thẳng vào thư mục đích (assets)
                    "ffmpeg", "-y", "-i", latest_tts,
                    "-c:a", "aac",
                    "-b:a", "64k",
                    "-ac", "1",
                    tts_audio_path
                ]
                # Chạy FFmpeg ngầm, tắt log rác
                subprocess.run(ffmpeg_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True)
                subprocess.run(["rm", latest_tts], check=True)  # Xóa file .wav gốc siêu nặng trong thư mục output
                print(f"  ✅ Đã lồng tiếng, NÉN (AAC 64k) và CUT file thành công: {tts_audio_path}")
            else:
                print(f"  ⚠️ Lỗi: Không tìm thấy file đầu ra TTS.")
        else:
            if os.path.exists(tts_audio_path):
                print(f"⏩ [2/4] Đã có sẵn file tts.wav. Bỏ qua lồng tiếng.")

        # ==========================================
        # BƯỚC 3: SAO CHÉP TÀI NGUYÊN LÊN GOOGLE DRIVE
        # ==========================================
        print(f"☁️ [3/4] Đang đồng bộ tài nguyên lên Google Drive...")
        # Lệnh Linux tạo folder đích (-p tự động tạo các folder cha nếu chưa có)
        subprocess.run(["mkdir", "-p", drive_assets_path], check=True)

        # Dùng lệnh Linux cp để copy chính xác 3 file (Dùng shell=True để nhận wildcard và nhiều param)
        cp_command = f"cp '{raw_srt_path}' '{sts_srt_path}' '{tts_audio_path}' '{drive_assets_path}/' 2>/dev/null || true"
        subprocess.run(cp_command, shell=True)

        print(f"  ✅ Đã sao chép 3 file tài nguyên cốt lõi vào: {drive_assets_path}")

print("\n🎉 HOÀN TẤT TOÀN BỘ CHUỖI XỬ LÝ!")

In [ ]:
#### 1a. Mute Audio

!uv run mute-srt \
    --input       /content/Survival/3/7509430509194890537.mp4 \
    --mute        /content/Survival/3/mute.srt \
    --output      /content/audio_muted.wav

In [ ]:
#### 1b. Extract Audio

!uv run extract-srt \
  --input /content/Survival/3/7509430509194890537.mp4 \
  --mute /content/Survival/3/mute.srt \
  --output /content/audio_extracted.wav

In [ ]:
#### 4a. Translate Note
!uv run translate-srt \
    --input  /content/Survival/3/note_source.srt \
    --output /content/note_translated.srt \
    --lang   "Japanese" \
    --keys   "{gemini_key}" \
    --model   "gemini-3.1-pro-preview" \
    --batch   100 \
    --wait    1.0

In [ ]:
#### 4b. Convert SRT to ASS

!uv run srt-to-ass \
    --input     /content/note_translated.srt \
    --output    /content/note_overlay.ass \
    --template  /content/CharenjiZukan/assets/sample.ass \
    --max-chars 14 \

In [ ]:
### Bước 5: Translate Subtitle
from google.colab import userdata
gemini_key = userdata.get('gemini_token')

!uv run cli/translate_srt.py \
    --input   /content/subtitle_merged.srt \
    --output  /content/subtitle_translated_long.srt \
    --lang    "Japanese" \
    --keys    "{gemini_key}" \
    --model   "gemini-3.1-pro-preview" \
    --batch   70 \
    --wait    1.0 \
    --verbose


In [ ]:
!pip show pyrubberband
!uv run media-speed --input /content/audio_bgm.m4a --speed 0.8

In [ ]:
### Bước 7: Slow Down 0.65x

# !uv run media-speed --input /content/Survival/3/7509430509194890537.mp4 --speed 0.65
# !uv run media-speed --input /content/audio_bgm.wav --speed 0.65
# !uv run media-speed --input /content/audio_extracted.wav --speed 0.65
!uv run media-speed --input /content/subtitle_translated.srt --speed 0.9
# !uv run media-speed --input /content/note_overlay.ass --speed 0.65


In [ ]:
### Bước 8: TTS

!uv run tts-srt --input /content/subtitle_translated_slow.srt --output /content/audioTTS_slow.wav --voice ja-JP-KeitaNeural --autorate --verbose

In [ ]:
# Nâng transformers cho Qwen3-VL
!uv pip install transformers

# Cài qwen-vl-utils (phiên bản khuyến nghị)
!uv pip install qwen-vl-utils

In [ ]:
!python run_colab_tests.py --tags sync_engine

In [ ]:
!uv run sync-video \
    --video /content/sample-extract.mp4 \
    --subtitle /content/sample-extract.srt \
    --black-bg /content/CharenjiZukan/assets/black-background.jpg \
    --note-overlay-png /content/CharenjiZukan/assets/note-overlay.png \
    --note-overlay-ass /content/sample.ass \
    --mute /content/mute.srt \
    --ambient "/content/Lonely-Dance.m4a" \
    --output-dir /content/output_sync \
    --tts-voice ja-JP-KeitaNeural \
    --workers 4 \
    --keep-tmp \
    --subtitle-fontname "Noto Sans CJK JP"

In [ ]:
!uv run sync-video \
    --video /content/Survival/3/7509430509194890537.mp4 \
    --subtitle /content/subtitle_translated_long.srt \
    --black-bg /content/CharenjiZukan/assets/black-background.jpg \
    --mute /content/mute.srt \
    --ambient "/content/Huma Huma Crimson Fly.mp3" \
    --output-dir /content/output_sync \
    --tts-voice ja-JP-KeitaNeural \
    --slow-cap 0.1 \
    --workers 4 \
    --subtitle-fontname "Noto Sans CJK JP"


In [14]:
!mv /content/output_sync/video_synced.mp4 /content/drive/MyDrive/CharenjiZukan/

In [ ]:
from google.colab import userdata
hf_token = userdata.get('hf_token')

!uv run video-ocr /content/Survival/3/7509430509194890537.mp4 \
    --output-dir /content/output \
    --config /content/CharenjiZukan/config/extractor_config.yaml \
    --boxes-file /content/CharenjiZukan/assets/boxesOCR.txt \
    --ocr-model Qwen/Qwen3-VL-8B-Instruct \
    --frame-interval 3 \
    --scene-threshold 20.0 \
    --min-scene-frames 3 \
    --max-duration 15.0 \
    --batch-size 8 \
    --device cuda \
    --hf-token "{hf_token}" \
    --warn-english \
    --format srt \
    -vv
    # --cv-prefilter \
